# Paper Story Board
- FDT/HMI cross calibration available but not easily applicable yet
- Synoptic map data set features periods where SDO is close to Earth and PHI produces maps that are almost identical in structure to the HMI maps
- direct pixel-to-pixel comparison in scatter plots as typical for cross calibration problems not applicable since to strict co-observation is available
  - observations are hours to about a day apart from the PHI orbit location
  - 2D histogram serves as example how no good correlation exists for this kind of analysis
  - 
- alternative way to assess the quality of combined synoptic maps necessary
  - flux balance and unsigned flux hint at the necessity of a cross calibration and somewhat confirm Alejandro et al
  - zero level analysis shows sub Gauss offsets, which are not enough to fix the flux balance
  - tinkering showed about 0.5 G necessary to balance wrt to HMI
  - flux balance apparently by applying the offset of Alejandro's cross calibration fit to filtered AR regions (at 250G seeds, 25 lower thld I think)
  - AR filtered pixels are about 10% of the total amount which at 5 G offset is about the 0.5 G found in above tinkering 
  - conclusion: cross calibration offset seems to provide the flux balance
  - total flux possibly corrected with cross calibration scaling but filtering for flux needs improvement
- AR flux filter
  - currently based on reconstruction from high seed to low threshold
  - try iterative version where we seed eg from 250->25, 200->25, 150->25 and combine all masks into a single mask that hopefully also selects for the lower overall magnetic activiy in PHI
  - check Burkhart Bovelet papers for multi level method used for bright points
  
- use 2290, 2291, 2293, 2294 in front of Earth and hopefully produce almost identical PFSS results
  - if PFSS is identical despite above quantification of the possible error, we can consider the PFSS results robust within error bars and discuss morphological differences in the farside observations
  - figure out if lack of decaying active region bananas in PHI makes a difference for PFSS (use differnet filtering)

- morphological differences in farside observations
  - possibly quantify with the geographical differences in magnetic activity
  - this can hopefully be done through the AR filter masks
- 

In [ ]:
import os, glob
import pandas as pd
import numpy as np
from scipy import optimize
from astropy.io import fits
import drms
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

from config.config import Config
from utils.plots import magnetic_flux_plot_latitudes, combined_synoptic_noise_plot, plot_synoptic_sources, plot_synoptic_with_stripe_magnitudes

In [ ]:
%matplotlib widget

In [ ]:

def ar_filtering(img, high_thld=250.0, low_thld=25.0, empty=0.0):

    from skimage.morphology import reconstruction

    seed = np.abs(img) >= high_thld
    mask = np.abs(img) >= low_thld

    final_mask = reconstruction(
        seed.astype(np.uint8),
        mask.astype(np.uint8),
        method='dilation'
    ).astype(bool)
    filtered_img = np.where(final_mask, img, empty)

    return filtered_img, final_mask

In [ ]:
def extract_cr(name: str) -> int:
    # name like "CR1234" or "CR1234_PHI"
    base = name.split("_")[0]    # → "CR1234"
    return int(base[2:])          # extract number

In [ ]:
# Complete CR 2280, 2283, 2284, 2285, 2286, 2287, 2291, 2293, 2294, 2296, 2298, 2299, 2300
# 2290, 2291, 2293, 2293 in front of Earth view
# HDIS: 
# 2290 = 2024.10.21_00:30:03_TAI-2024.11.12_18:15:03_TAI = ~0.6 AU
# 2291 = 2024.11.12_21:15:03_TAI-2025.01.05_15:15:09_TAI = ~0.8 AU
# 2293 = 2025.01.05_18:15:09_TAI-2025.02.01_12:15:09_TAI = ~0.8 AU
# 2294 = 2025.02.01_15:45:09_TAI-2025.03.01_06:15:09_TAI = ~0.7 AU 
carrington_number = 2291

In [ ]:
root = Path.cwd()
#path = Path('/scratch/slam/loeschl/dev/python/synoptic-map-pipeline/output/release_2025_v01/l3/syn/PHIHMI/CR%s/synop/'%carrington_number)
path = Path('/scratch/slam/loeschl/dev/python/synoptic-map-pipeline/output/release_2025_v01/l3/syn/PHI/CR%s_PHI/synop/'%carrington_number)

#fname = "synopMr.fits"
#series="hmi.synoptic_mr_polfil_720s"
#segment="Mr_polfil"
#small = False

fname   = "synopMr_small.fits"
series  = "hmi.mrsynop_small_720s"
segment = "synopMr"
small   = True

In [ ]:
############################
####### GET PHI DATA #######
############################

synop_phi  = fits.open(path / fname)

phi_img   = synop_phi[0].data
if not small: 
    phi_table = synop_phi[1].data

# Define Carrington rotation number
#carrington_number = int(synop_phi[0].header["CAR_ROT"])
outname = f"{carrington_number}"


############################
####### GET HMI DATA #######
############################

# Create DRMS client
c = drms.Client(email="loeschl@mps.mpg.de", verbose=True)

datapath_hmi = os.path.join(root, f"../data/tmp/CR{carrington_number}/")
os.makedirs(datapath_hmi, exist_ok=True)

# Query JSOC for that rotation
q = c.query(f"{series}[{carrington_number}]", seg=segment)
fname_hmi = q[segment][0].split('/')[-1]

try:
    file_hmi  = glob.glob(f"{datapath_hmi}*{fname_hmi}")[0]
    synop_hmi = fits.open(file_hmi)
except IndexError:
    # Download the FITS file
    result = c.export(f"{series}[{carrington_number}]", method='url', protocol='fits')
    result.download(datapath_hmi)
    file_hmi  = glob.glob(f"{datapath_hmi}*{fname_hmi}")[0]
    synop_hmi = fits.open(file_hmi)

try: 
    hmi_img = synop_hmi[1].data   

    # create mask of NaN values in phi_img and fill them with hmi_img values
    mask_polfil = np.isnan(phi_img)
    phi_polfil  = np.where(mask_polfil, hmi_img, phi_img)    

    # aggressive HMI pole filling
    #phi_polfil[:40, :]  = hmi_img[:40, :]
    #phi_polfil[1400: :] = hmi_img[1400:, :]

    #synop_phi[0].header["CUNIT2"] = "deg" # "Sine Latitude"

    primary_hdu = fits.PrimaryHDU()

    polfil_hdu = fits.CompImageHDU(data=phi_polfil, 
                                    header=synop_phi[0].header, 
                                    compression_type='RICE_1')

    polfil_hdul = fits.HDUList([primary_hdu, polfil_hdu])
    polfil_hdul.writeto(os.path.join(path, "synopMr_polfil.fits"), overwrite=True)

    # show filled synoptic map 
    #plt.imshow(phi_polfil, cmap='hmimag', vmin=-1500, vmax=1500, origin='lower')
    #plt.show()

except IndexError:
    # non polfil data
    hmi_img = synop_hmi[0].data

# populate config for diagnostics plots
config = Config(path / "../config.yaml")
#config.cr = carrington_number
config.update("cr", carrington_number)

if "Mr" in segment:
    config.Mr = True
    config.Btype = "Radial"
else:
    config.Mr = False
    config.Btype = "line-of-sight"

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 8), nrows=2, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_img, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI CR')
im2 = ax[1].imshow(hmi_img, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='HMI CR')

ax[0].set_title('PHI CR %s'%carrington_number)
ax[1].set_title('HMI CR %s'%carrington_number)

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')

## Active Region Filtering 

In [ ]:
high_thld = 100.0
low_thld  = 25.0

In [ ]:
phi_ar, phi_ar_mask = ar_filtering(phi_img, high_thld=high_thld, low_thld=low_thld, empty=np.nan)
phi_ar = np.where(phi_ar_mask, phi_img, np.nan)

hmi_ar, hmi_ar_mask = ar_filtering(hmi_img, high_thld=high_thld, low_thld=low_thld, empty=np.nan)
hmi_ar = np.where(hmi_ar_mask, hmi_img, np.nan)


In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 8), nrows=2, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_ar, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI')
im2 = ax[1].imshow(hmi_ar, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='HMI')

ax[0].set_title('PHI Active Regions CR %s'%carrington_number)
ax[1].set_title('HMI Active Regions CR %s'%carrington_number)

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')

In [ ]:
# todo remove
phi_ar_off = phi_ar + 5
hmi_pos = hmi_ar[hmi_ar>=0].flatten().sum()
hmi_neg = hmi_ar[hmi_ar<0].flatten().sum()

phi_pos_off = phi_ar_off[phi_ar_off>=0].flatten().sum()
phi_neg_off = phi_ar_off[phi_ar_off<0].flatten().sum()    

hmi_ar_unsigned = np.nansum(np.abs(hmi_ar).flatten())
phi_ar_off_unsigned = np.nansum(np.abs(phi_ar_off).flatten())

plt.close()
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative', 'HMI unsigned', 'PHI unsigned'], [hmi_pos, abs(hmi_neg), phi_pos_off, abs(phi_neg_off), hmi_ar_unsigned, phi_ar_unsigned])
ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Map AR") 


In [ ]:
hmi_ar_unsigned / phi_ar_off_unsigned

In [ ]:
hmi_pos = hmi_ar[hmi_ar>=0].flatten().sum()
hmi_neg = hmi_ar[hmi_ar<0].flatten().sum()

phi_pos = phi_ar[phi_ar>=0].flatten().sum()
phi_neg = phi_ar[phi_ar<0].flatten().sum()    

hmi_ar_unsigned = np.nansum(np.abs(hmi_ar).flatten())
phi_ar_unsigned = np.nansum(np.abs(phi_ar).flatten())

plt.close()
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative', 'HMI unsigned', 'PHI unsigned'], [hmi_pos, abs(hmi_neg), phi_pos, abs(phi_neg), hmi_ar_unsigned, phi_ar_unsigned])
ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Map AR") 


In [ ]:
# find number of non-nan pixels in both maps
n_hmi_ar = len(np.abs(hmi_ar[~np.isnan(hmi_ar)]).flatten())
n_phi_ar = len(np.abs(phi_ar[~np.isnan(phi_ar)]).flatten())

In [ ]:
n_phi_ar/(3600*1440)

In [ ]:
# compute average flux density in both maps
flux_dens_hmi  = hmi_ar_unsigned/n_hmi_ar
flux_dens_phi = phi_ar_unsigned/n_phi_ar
flux_dens_hmi, flux_dens_phi

In [ ]:
flux_dens_hmi/flux_dens_phi, flux_dens_phi/flux_dens_hmi

In [ ]:
hmi_ar_unsigned/phi_ar_unsigned, phi_ar_unsigned/hmi_ar_unsigned

In [ ]:
hmi_pos = hmi_img[hmi_img>=0].flatten().sum()
hmi_neg = hmi_img[hmi_img<0].flatten().sum()

phi_pos = phi_img[phi_img>=0].flatten().sum()
phi_neg = phi_img[phi_img<0].flatten().sum()    

hmi_unsigned = np.nansum(np.abs(hmi_img).flatten())
phi_unsigned = np.nansum(np.abs(phi_img).flatten())

plt.close()
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative', 'HMI unsigned', 'PHI unsigned'], [hmi_pos, abs(hmi_neg), phi_pos, abs(phi_neg), hmi_unsigned, phi_unsigned])
ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Maps") 


In [ ]:
# find number of non-nan pixels in both maps
n_hmi = len(np.abs(hmi_img[~np.isnan(hmi_img)]).flatten())
n_phi = len(np.abs(phi_img[~np.isnan(phi_img)]).flatten())

In [ ]:
# compute average flux density in both maps
flux_dens_hmi  = hmi_unsigned/n_hmi
flux_dens_phi = phi_unsigned/n_phi  
flux_dens_hmi, flux_dens_phi

In [ ]:
flux_dens_hmi/flux_dens_phi, flux_dens_phi/flux_dens_hmi

In [ ]:
hmi_unsigned/phi_unsigned, phi_unsigned/hmi_unsigned

In [ ]:
phi_img_off = phi_img + 0.5

hmi_pos = hmi_img[hmi_img>=0].flatten().sum()
hmi_neg = hmi_img[hmi_img<0].flatten().sum()

phi_pos = phi_img_off[phi_img_off>=0].flatten().sum()
phi_neg = phi_img_off[phi_img_off<0].flatten().sum()    

hmi_unsigned = np.nansum(np.abs(hmi_img).flatten())
phi_unsigned = np.nansum(np.abs(phi_img_off).flatten())

plt.close()
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative', 'HMI unsigned', 'PHI unsigned'], [hmi_pos, abs(hmi_neg), phi_pos, abs(phi_neg), hmi_unsigned, phi_unsigned])
ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Maps") 


In [ ]:
plt.close()
nbins = 250

fig, ax = plt.subplots(figsize=(8,6))

n, bin, patches = ax.hist(hmi_ar.flatten(), bins=nbins, density=False, label="HMI AR filtered")
n, bin, patches = ax.hist(phi_ar.flatten(), bins=nbins, density=False, label="PHI AR filtered")

ax.set_yscale('log')
ax.set_xlabel("Magnetic Field [G]")
ax.set_ylabel("Counts") 

plt.legend()



In [ ]:
plt.close()
nbins = 250

fig, ax = plt.subplots(figsize=(8,6))

n, bin, patches = ax.hist(hmi_img.flatten(), bins=nbins, density=False, label="HMI")
n, bin, patches = ax.hist(phi_img.flatten(), bins=nbins, density=False, label="PHI")

ax.set_yscale('log')
ax.set_xlabel("Magnetic Field [G]")
ax.set_ylabel("Counts") 

plt.legend()



## Active Region Window

In [ ]:
# Select data segment
x1 = 0
x2 = 3600
y1 = 200
y2 = 1240

hmi_seg = hmi_img[y1:y2, x1:x2] # hmi standard synoptic
phi_seg = phi_img[y1:y2, x1:x2] # hmi standard synoptic

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(6,6), nrows=2, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_seg, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI')
im2 = ax[1].imshow(hmi_seg, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='HMI')

ax[0].set_title('PHI Active Regions CR %s'%carrington_number)
ax[1].set_title('HMI Active Regions CR %s'%carrington_number)

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')

In [ ]:
from matplotlib.colors import LogNorm

####### DENSITY PLOTS ########

vmin = -2000
vmax = +2000

bins = 160

fig, ax = plt.subplots(figsize=(6,5))   
counts, xedges, yedges, im = ax.hist2d(phi_seg.flatten(), hmi_seg.flatten(), bins=(bins, bins), range=((vmin, vmax), (vmin, vmax)), norm=LogNorm(), cmap="cividis")


In [ ]:
fig, (ax1, ax2)= plt.subplots(figsize=(15, 6), nrows=1, ncols=2, sharex=True, sharey=True)
ax1.scatter(phi_img.flatten(), hmi_img.flatten(), s=0.1)
ax1.set_xlim(-2000, 2000)

ax2.scatter(phi_ar.flatten(), hmi_ar.flatten(), s=0.1)
ax2.set_xlim(-2000, 2000)

ax1.set_xlabel("PHI Magnetic Field [G]")
ax1.set_ylabel("HMI Magnetic Field [G]")

ax2.set_xlabel("PHI Magnetic Field [G]")
ax2.set_ylabel("HMI Magnetic Field [G]")

In [ ]:
phi_ar_rev = np.where(~phi_ar_mask, phi_img, np.nan)
hmi_ar_rev = np.where(~hmi_ar_mask, hmi_img, np.nan)

common_mask = phi_ar_mask & hmi_ar_mask
phi_ar_rev2 = np.where(~common_mask, phi_img, np.nan)
hmi_ar_rev2 = np.where(~common_mask, hmi_img, np.nan)


In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(6,6), nrows=2, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_ar_rev2, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI')
im2 = ax[1].imshow(hmi_ar_rev2, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='HMI')

ax[0].set_title('PHI Active Regions CR %s'%carrington_number)
ax[1].set_title('HMI Active Regions CR %s'%carrington_number)

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')

In [ ]:
bins = 400

fig, (ax1, ax2, ax3)= plt.subplots(figsize=(20, 6), nrows=1, ncols=3, sharex=True, sharey=True)
counts1, xedges1, yedges1, im1 = ax1.hist2d(phi_img.flatten(), hmi_img.flatten(), bins=(bins, bins), range=((vmin, vmax), (vmin, vmax)), norm=LogNorm(), cmap="cividis")

ax1.set_xlim(-2000, 2000)
ax1.set_ylim(-2000, 2000)


counts2, xedges2, yedges2, im2 = ax2.hist2d(phi_ar.flatten(), hmi_ar.flatten(), bins=(bins, bins), range=((vmin, vmax), (vmin, vmax)), norm=LogNorm(), cmap="cividis")
ax2.set_xlim(-2000, 2000)
ax2.set_ylim(-2000, 2000)

counts3, xedges3, yedges3, im3 = ax3.hist2d(phi_ar_rev2.flatten(), hmi_ar_rev2.flatten(), bins=(bins, bins), range=((vmin, vmax), (vmin, vmax)), norm=LogNorm(), cmap="cividis")
ax3.set_xlim(-2000, 2000)
ax3.set_ylim(-2000, 2000)


ax1.set_xlabel("PHI Magnetic Field [G]")
ax1.set_ylabel("HMI Magnetic Field [G]")

ax2.set_xlabel("PHI Magnetic Field [G]")
ax2.set_ylabel("HMI Magnetic Field [G]")

In [ ]:
hmi_pos = hmi_seg[hmi_seg>=0].flatten().sum()
hmi_neg = hmi_seg[hmi_seg<0].flatten().sum()

phi_pos = phi_seg[phi_seg>=0].flatten().sum()
phi_neg = phi_seg[phi_seg<0].flatten().sum()

hmi_unsigned = np.nansum(np.abs(hmi_seg).flatten())
phi_unsigned = np.nansum(np.abs(phi_seg).flatten())

plt.close()
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative', 'HMI unsigned', 'PHI unsigned'], [hmi_pos, abs(hmi_neg), phi_pos, abs(phi_neg), hmi_unsigned, phi_unsigned])
ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Maps")  


In [ ]:
phi_unsigned/hmi_unsigned

In [ ]:
hmi_unsigned/phi_unsigned

In [ ]:
hmi_pos = hmi_img[hmi_img>=0].flatten().sum()
hmi_neg = hmi_img[hmi_img<0].flatten().sum()

phi_pos = phi_img[phi_img>=0].flatten().sum()
phi_neg = phi_img[phi_img<0].flatten().sum()    

hmi_unsigned = np.nansum(np.abs(hmi_img).flatten())
phi_unsigned = np.nansum(np.abs(phi_img).flatten())

plt.close()
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative', 'HMI unsigned', 'PHI unsigned'], [hmi_pos, abs(hmi_neg), phi_pos, abs(phi_neg), hmi_unsigned, phi_unsigned])
ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Maps") 


## Active region removal test

In [ ]:
high_thld = 50.0
low_thld  = 5.0

In [ ]:
plt.close()
phi_filtered, phi_mask = ar_filtering(phi_img, high_thld=high_thld, low_thld=low_thld, empty=np.nan)
phi_filtered_rev = np.where(~phi_mask, phi_img, np.nan)
phi_filtered_rev = np.where(np.abs(phi_filtered_rev) <=10, phi_filtered_rev, np.nan)
plt.imshow(phi_filtered_rev, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None')

In [ ]:
plt.close()
phi_filtered_rev2 = np.where(np.abs(phi_img) <=10, phi_img, np.nan)
plt.imshow(phi_filtered_rev2, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None')

In [ ]:
plt.close()
hmi_filtered, hmi_mask = ar_filtering(hmi_img, high_thld=high_thld, low_thld=low_thld, empty=np.nan)
plt.imshow(hmi_filtered, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='none')